<a href="https://colab.research.google.com/github/Ryan-T-Monash/MMA3001-Project/blob/main/notebooks/Investigating_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
# Pick which NASA IMS bearing files to use. Paste into a Colab cell and run.
# The cell finishes straight away; your choice is saved in `selected` when you press Confirm.
import os
import ipywidgets as w
from IPython.display import display

selected = []
folder_box = w.Text(value="/content/drive/MyDrive/MMA3001/MMA3001_Project/IMS/2nd_test", description="Folder:")
folder_btn = w.Button(description="Confirm folder")
file_list = w.SelectMultiple(rows=15, description="Files:")
files_btn = w.Button(description="Confirm files")
msg = w.Label()

def confirm_folder(_):
    if os.path.isdir(folder_box.value):
        # skip Drive duplicates like "2004.02.19.05.22 (1).39"; timestamps sort in time order
        file_list.options = sorted(f for f in os.listdir(folder_box.value) if " (" not in f)
        msg.value = f"{len(file_list.options)} files found. Ctrl/Shift-click to choose."
    else:
        msg.value = "Folder not found, check the path."

def confirm_files(_):
    selected[:] = [os.path.join(folder_box.value, f) for f in file_list.value]
    msg.value = f"Selected {len(selected)} files. You can run the next cell."

folder_btn.on_click(confirm_folder)
files_btn.on_click(confirm_files)
display(w.HBox([folder_box, folder_btn]), file_list, files_btn, msg)


SelectMultiple(description='Files:', options=(), rows=15, value=())

Button(description='Confirm files', style=ButtonStyle())

Label(value='')

In [18]:
import numpy as np
import pandas as pd
import os

rows = []
for path in [p for p in selected if " (" not in os.path.basename(p)]:
    data = np.loadtxt(path)              # one snapshot: rows = samples, columns = bearing channels
    row = {"time": pd.to_datetime(os.path.basename(path), format="%Y.%m.%d.%H.%M.%S")}
    for ch in range(data.shape[1]):
        x = data[:, ch]
        row[f"rms_{ch+1}"] = np.sqrt(np.mean(x**2))
        row[f"kurtosis_{ch+1}"] = np.mean((x - x.mean())**4) / np.var(x)**2
        row[f"peak_{ch+1}"] = np.max(np.abs(x))
    rows.append(row)

features = pd.DataFrame(rows).set_index("time")
features.to_csv("/content/drive/MyDrive/MMA3001/MMA3001_Project/features_2nd_test.csv")
features.head()

,rms_1,kurtosis_1,peak_1,rms_2,kurtosis_2,peak_2,rms_3,kurtosis_3,peak_3,rms_4,kurtosis_4,peak_4
time,,,,,,,,,,,,
2004-02-12 10:32:39,0.074179,3.628763,0.454,0.090944,3.506801,0.513,0.109404,6.213075,1.023,0.054103,3.065959,0.264
2004-02-12 10:42:39,0.075382,3.648291,0.388,0.093419,3.253014,0.481,0.109817,4.395250,0.752,0.056101,3.107539,0.249
2004-02-12 10:52:39,0.076230,3.513475,0.503,0.093718,3.310789,0.503,0.109861,5.639948,0.933,0.056145,3.257236,0.354
2004-02-12 11:02:39,0.078724,4.157953,0.608,0.092947,3.235341,0.569,0.110667,5.682779,1.016,0.056807,3.806190,0.525
2004-02-12 11:12:39,0.078474,3.603177,0.391,0.095348,3.226309,0.486,0.107506,4.578395,0.771,0.056841,3.138954,0.244
